# Hook demo (facilitator only)
### Agenda item 1, 00:00-00:08. Record the skit, press one button, show the minutes.

For the person running the hook. Participants don't use this file. Their notebook is `audio_workshop.ipynb`.

Run section 1 before the session. It loads the Whisper model, which takes a minute or two. Don't do it live.

During the hook:
1. Section 2: record the skit from the browser mic.
2. Section 3: press the button on that recording.
3. Section 4: use the backup WAV if recording fails.
4. Section 5: text-only fallback if Whisper won't load.

The live skit is not rehearsed or tested end to end. Only the backup WAV is verified. Rehearse the live path first (checklist in `demo/FACILITATOR.md`). Until then, use the backup WAV.

Pacing for a live skit:
- Say each turn in one go. No long pause inside it.
- Two turns have two sentences. Say both without stopping.
- Leave a clear 2-second pause before the other person speaks.
- Don't overlap.

The transcriber starts a new speaker at a 1.5-second gap. A quick back-and-forth merges into one speaker. A long pause inside a turn looks like a speaker change.

What to say and when: `demo/FACILITATOR.md`.

## 1. Setup: run before the session
On Colab: Runtime > Change runtime type > T4 GPU. It makes Whisper faster.
Upload `facilitator_files.zip` to the Files panel (left) first.

In [ ]:
import os, sys, shutil, subprocess, importlib.util, zipfile
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import warnings; warnings.filterwarnings("ignore", message="FP16")  # Whisper on CPU only

if importlib.util.find_spec("whisper") is None:
    print("Installing Whisper, about a minute...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai-whisper"], check=True)
if os.path.exists("facilitator_files.zip") and not os.path.exists("prototype"):
    zipfile.ZipFile("facilitator_files.zip").extractall(".")
PROTO = next((p for p in ("prototype", "../prototype") if os.path.isdir(p)), None)
PROTO or sys.exit("Upload facilitator_files.zip to the Files panel (left), then run this cell again.")
sys.path.insert(0, PROTO)
print("ffmpeg:", "found" if shutil.which("ffmpeg") else "NOT FOUND")

# medium.en is fast on a T4. On CPU it takes about 3 minutes (measured), so use small.en:
# same minutes, about 1 minute.
import torch
has_gpu = torch.cuda.is_available()
print(f"GPU: {'yes' if has_gpu else 'no'} -- expect the button press to take "
      f"{'under a minute' if has_gpu else 'about 1 minute (small.en, no GPU here)'}")
os.environ["WHISPER_MODEL"] = "medium.en" if has_gpu else "small.en"

# Preload now, not during the hook. The first Whisper call downloads and loads the model.
import whisper
import transcriber
transcriber._model = whisper.load_model(transcriber.WHISPER_MODEL)
print(f"Whisper ({transcriber.WHISPER_MODEL}) preloaded and ready.")

# Cells 6, 8 and 10 need these. Imported once here, so each cell works on its own.
from demo import run_demo
from IPython.display import Audio, HTML, Markdown, display


def big_minutes(minutes_markdown):
    """Show generate_minutes() output as large HTML for a projector.
    meeting_minutes.py is frozen, so this only enlarges its output."""
    import html as html_lib
    lines_out = ['<div style="font-size:160%; line-height:1.5">']
    for line in minutes_markdown.splitlines():
        text = html_lib.escape(line)
        if line.startswith("# "):
            lines_out.append(f"<h1>{text[2:]}</h1>")
        elif line.startswith("## "):
            lines_out.append(f"<h2>{text[3:]}</h2>")
        elif line.startswith("- [ ] "):
            lines_out.append(f"<div>&#9744; {text[6:]}</div>")
        elif line.startswith("- "):
            lines_out.append(f"<div>&bull; {text[2:]}</div>")
        elif line.startswith("**") and line.endswith("**") is False and "**" in line:
            bold_text = text.replace("**", "<b>", 1).replace("**", "</b>", 1)
            lines_out.append(f"<p>{bold_text}</p>")
        elif line.strip():
            lines_out.append(f"<p>{text}</p>")
    lines_out.append("</div>")
    return HTML(chr(10).join(lines_out))


## 2. Record the skit from the browser microphone
Click record. Do the skit (two presenters, `demo/script.txt`). Say each turn in one go. Pause 2 seconds before the other person speaks (see the pacing note above). The recording saves to `skit.wav`.

Needs Colab. On a laptop, record with any app and skip to section 3 with that file's path.

In [ ]:
from google.colab import output
from google.colab.output import eval_js
from base64 import b64decode
from IPython.display import Javascript, display
import subprocess

RECORD_JS = """
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = () => resolve(reader.result);
  reader.readAsDataURL(blob);
});
var recorder, chunks = [], micStream;
async function startRecording() {
  micStream = await navigator.mediaDevices.getUserMedia({audio: true});
  recorder = new MediaRecorder(micStream);
  chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
}
async function stopRecording() {
  recorder.stop();
  await new Promise(resolve => recorder.onstop = resolve);
  micStream.getTracks().forEach(t => t.stop());  // release the mic / turn off the browser's recording indicator
  const blob = new Blob(chunks);
  return await b2text(blob);
}
"""

def record_skit(out_path: str = "skit.wav") -> str:
    """Browser mic to out_path (WAV). Recording starts at once. Press Enter at the prompt to stop.
    Two plain calls, not a widget: more reliable across Colab versions."""
    display(Javascript(RECORD_JS))
    eval_js("startRecording()")
    input("Recording... press Enter here when the skit is done. ")
    data_url = eval_js("stopRecording()")
    _, encoded = data_url.split(",", 1)
    webm_path = out_path.replace(".wav", ".webm")
    with open(webm_path, "wb") as f:
        f.write(b64decode(encoded))
    # ffmpeg converts the browser's webm to WAV.
    result = subprocess.run(["ffmpeg", "-y", "-i", webm_path, out_path], capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)   # show why ffmpeg failed
        raise RuntimeError(f"ffmpeg failed converting {webm_path} -> {out_path} (see stderr above)")
    print(f"Saved {out_path}")
    return out_path

skit_path = record_skit()


## 3. Press. One. Button. (the live recording)

In [ ]:
minutes = run_demo(skit_path, banner=False)
display(big_minutes(minutes))


## 4. Backup: the two-voice WAV
Use this if the recording failed. See `demo/make_backup_wav.py` for how it was made and `demo/FACILITATOR.md` for when to use it.

Two cells. Run them in order, with a pause between:
1. The next cell plays the WAV to the room.
2. The cell under 4b presses the button.

The room calls out the decisions and to-dos before the minutes appear.

In [ ]:
backup_path = next(p for p in ("demo/backup_meeting.wav", "../demo/backup_meeting.wav") if os.path.isfile(p))
display(Audio(backup_path, autoplay=False))   # play this to the room first.
# Let them call out the decisions and to-dos before the next cell presses the button.


## 4b. Press the button on the backup
Run this only after the room has called out the decisions and to-dos.


In [ ]:
minutes = run_demo(backup_path, banner=False)
display(big_minutes(minutes))


## 5. Text-only fallback
Use this if Whisper won't load. It runs `demo/script.txt` as typed text through the same minutes code. No audio, no model, instant.

It isn't the real skit, but it keeps the hook going.

In [ ]:
minutes = run_demo(banner=False)   # no argument: demo/script.txt, no Whisper at all
display(big_minutes(minutes))
